In [ ]:
# Importações comuns para a maioria dos exemplos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Configurações para os plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

# Clustering

## K-Means passo a passo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.metrics.pairwise import euclidean_distances
import seaborn as sns

from sklearn.cluster import KMeans

In [ ]:
def plot_kmeans_iteration(X, centroids, assignments, iteration, title_suffix=""):
    """
    Plota o estado atual do K-Means: pontos, clusters e centróides.
    """
    plt.figure(figsize=(10, 7))

    # Cores para os clusters
    unique_clusters = np.unique(assignments)
    # Garantir que temos cores suficientes, mesmo que alguns clusters fiquem vazios temporariamente
    # Usaremos um mapa de cores padrão do matplotlib
    colors = plt.cm.get_cmap('viridis', max(len(unique_clusters), len(centroids)))

    for k_idx, k in enumerate(unique_clusters):
        if k == -1: # Caso algum ponto não seja atribuído (não deveria acontecer em K-Means padrão)
            cluster_points = X[assignments == k]
            plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
                        s=50, color='grey', label='Não atribuído', alpha=0.5)
        else:
            cluster_points = X[assignments == k]
            # Usar k como índice se for numérico e dentro da faixa de cores
            # Se k for um rótulo que pode não ser um índice válido, mapeie para um índice
            color_idx = k_idx if k_idx < colors.N else 0
            plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
                        s=70, color=colors(color_idx / (colors.N -1 if colors.N > 1 else 1) ),
                        label=f'Cluster {int(k)+1}', alpha=0.7)

    # Plotar os centróides
    plt.scatter(centroids[:, 0], centroids[:, 1],
                s=250, marker='X', color='red',
                edgecolor='black', linewidth=1.5, label='Centróides')

    plt.title(f'K-Means: Iteração {iteration} {title_suffix}')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')

    # Ajustar a legenda para não ter muitas entradas se K for grande
    handles, labels = plt.gca().get_legend_handles_labels()
    # Remover legendas duplicadas de clusters se houver
    by_label = dict(zip(labels, handles))
    # Garantir que a legenda dos centróides apareça
    if 'Centróides' not in by_label and any(label.startswith('Cluster') for label in by_label):
         # Adicionar centróides manualmente se não estiverem, mas houver clusters
        pass # O scatter dos centróides já adiciona a legenda

    plt.legend(by_label.values(), by_label.keys(), loc='best')
    plt.grid(True)
    plt.show()

In [ ]:
def kmeans_step_by_step(X, n_clusters, max_iters=10, random_state=None):
    """
    Executa o K-Means passo a passo e plota cada iteração.
    """
    if random_state:
        np.random.seed(random_state)

    # 1. Inicializar centróides aleatoriamente a partir dos pontos de dados
    # Isso garante que os centróides estejam dentro da faixa dos dados
    initial_indices = np.random.choice(X.shape[0], size=n_clusters, replace=False)
    centroids = X[initial_indices, :]

    print(f"Centróides Iniciais:\n{centroids}")
    # Plot inicial com os centróides escolhidos e sem atribuições ainda
    # Para a plotagem inicial, podemos atribuir todos os pontos a um cluster dummy ou não plotar as atribuições
    initial_assignments = np.zeros(X.shape[0], dtype=int) # Dummy assignments
    plot_kmeans_iteration(X, centroids, initial_assignments, 0, "(Após Inicialização dos Centróides)")

    # Loop de iterações
    for i in range(max_iters):
        # 2. Passo de Atribuição: Atribuir cada ponto ao centróide mais próximo
        distances = euclidean_distances(X, centroids) # Matriz de distâncias (n_samples, n_clusters)
        assignments = np.argmin(distances, axis=1) # Índice do centróide mais próximo para cada ponto

        plot_kmeans_iteration(X, centroids, assignments, i + 1, "(Após Atribuição)")

        # 3. Passo de Atualização: Recalcular os centróides
        new_centroids = np.zeros_like(centroids)
        clusters_updated = False
        for k in range(n_clusters):
            cluster_points = X[assignments == k]
            if len(cluster_points) > 0:
                new_centroids[k, :] = cluster_points.mean(axis=0)
            else:
                # Se um cluster ficar vazio, re-inicialize seu centróide
                # (pode acontecer, embora raro com boa inicialização e dados)
                # Ou mantenha o centróide antigo, ou escolha um ponto aleatório
                print(f"Atenção: Cluster {k+1} ficou vazio na iteração {i+1}. Mantendo centróide anterior ou re-inicializando.")
                # Para simplificar, vamos manter o centróide anterior se estiver vazio.
                # Uma estratégia melhor seria re-inicializar para um ponto distante.
                new_centroids[k, :] = centroids[k, :]


        plot_kmeans_iteration(X, new_centroids, assignments, i + 1, "(Após Atualização dos Centróides)")

        # Se os centróides não mudaram, o algoritmo convergiu
        if np.allclose(centroids, new_centroids, atol=1e-4): # Usar atol para tolerância
            print(f"\nConvergência alcançada na iteração {i+1}.")
            centroids = new_centroids # Atualizar para a plotagem final
            clusters_updated = True # Para garantir que a plotagem final ocorra
            break

        centroids = new_centroids
        if i == max_iters - 1:
            print("\nNúmero máximo de iterações atingido.")
            clusters_updated = True

    # Plot final
    if clusters_updated:
        final_distances = euclidean_distances(X, centroids)
        final_assignments = np.argmin(final_distances, axis=1)
        plot_kmeans_iteration(X, centroids, final_assignments, "Final", "(Convergido)")

    return centroids, final_assignments

In [ ]:
# --- Geração de Dados de Exemplo ---
N_SAMPLES = 300
N_FEATURES = 2
N_CLUSTERS_TRUE = 3 # Número real de clusters nos dados gerados
RANDOM_STATE_DATA = 42

In [ ]:
X_blobs, y_blobs_true = make_blobs(n_samples=N_SAMPLES,
                                   n_features=N_FEATURES,
                                   centers=N_CLUSTERS_TRUE,
                                   cluster_std=1.0, # Desvio padrão dos clusters
                                   random_state=RANDOM_STATE_DATA)

In [ ]:
# Plotar os dados originais para referência
plt.figure(figsize=(10, 7))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], color='black', s=50, alpha=0.7)
plt.title('Dados Originais Gerados (Ground Truth)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True)
plt.show()


In [ ]:
K_TO_FIND = 3 # Número de clusters que o K-Means tentará encontrar
MAX_ITERATIONS = 10 # Número máximo de iterações para visualização
RANDOM_STATE_KMEANS = 10 # Para reprodutibilidade da inicialização dos centróides

In [ ]:
print(f"Iniciando K-Means para encontrar K={K_TO_FIND} clusters...")
final_centroids, final_assignments = kmeans_step_by_step(X_blobs,
                                                         n_clusters=K_TO_FIND,
                                                         max_iters=MAX_ITERATIONS,
                                                         random_state=RANDOM_STATE_KMEANS)

print(f"\nCentróides Finais Encontrados:\n{final_centroids}")
print(f"\nAtribuições Finais dos Pontos:\n{final_assignments}")

## Definição de quantidade de Ks - Método do Cotovelo (Elbow Method)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.metrics.pairwise import euclidean_distances
import seaborn as sns

from sklearn.cluster import KMeans

In [ ]:
dados = pd.read_csv('/content/alimentos.csv', sep=";", encoding='latin-1')

In [ ]:
dados.head()

In [ ]:
X_dados = dados.drop('Descrição dos alimentos', axis=1).values
print(X_dados)

In [ ]:
#Elbow Method (Método do Cotovelo)
inertia_values = []
possible_k_values = range(1, 11)
for k_val in possible_k_values:
    kmeans_temp = KMeans(n_clusters=k_val, init='k-means++', n_init='auto', random_state=42)
    kmeans_temp.fit(X_dados)
    inertia_values.append(kmeans_temp.inertia_)

In [ ]:
#Elbow Method (Método do Cotovelo)
plt.figure(figsize=(10, 7))
plt.plot(possible_k_values, inertia_values, marker='o', linestyle='-')
plt.title('Método do cotovelo (escolha de clusters)')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inércia (WCSS)')
plt.xticks(possible_k_values)
plt.grid(axis='y')
plt.show()

In [ ]:
# Plotar os dados originais para referência
plt.figure(figsize=(10, 7))
plt.scatter(X_dados[:, 0], X_dados[:, 1], color='black', s=50, alpha=0.7)
plt.title('Dados Originais Gerados')
plt.xlabel('Lipídeos')
plt.ylabel('Ferro')
plt.grid(True)
plt.show()

In [ ]:
K_TO_FIND = 4
MAX_ITERATIONS = 10
RANDOM_STATE_KMEANS = 42

In [ ]:
print(f"Iniciando K-Means para encontrar K={K_TO_FIND} clusters...")
final_centroids, final_assignments = kmeans_step_by_step(X_dados,
                                                         n_clusters=K_TO_FIND,
                                                         max_iters=MAX_ITERATIONS,
                                                         random_state=RANDOM_STATE_KMEANS)

print(f"\nCentróides Finais Encontrados:\n{final_centroids}")
print(f"\nAtribuições Finais dos Pontos:\n{final_assignments}")

In [ ]:
dados.info()
print(dados)

## K-Means

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [ ]:
url_seeds = "https://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt"
column_names_seeds = ['area', 'perimeter', 'compactness', 'length_kernel', 'width_kernel', 'asymmetry_coefficient', 'length_kernel_groove', 'target']
df_seeds = pd.read_csv(url_seeds, sep='\s+', header=None, names=column_names_seeds)

In [ ]:
df_seeds.info()
df_seeds.describe()

In [ ]:
X_seeds = df_seeds.drop('target', axis=1).values
y_seeds_true = df_seeds['target'].values

In [ ]:
#Padronização da escala
scaler_seeds = StandardScaler()
X_seeds_scaled = scaler_seeds.fit_transform(X_seeds)

In [ ]:
#Elbow Method (Método do Cotovelo)
inertia_values = []
possible_k_values = range(1, 11)
for k_val in possible_k_values:
    kmeans_temp = KMeans(n_clusters=k_val, init='k-means++', n_init='auto', random_state=42)
    kmeans_temp.fit(X_seeds_scaled)
    inertia_values.append(kmeans_temp.inertia_)

In [ ]:
#Elbow Method (Método do Cotovelo)
plt.figure(figsize=(10, 7))
plt.plot(possible_k_values, inertia_values, marker='o', linestyle='-')
plt.title('Método do cotovelo (escolha de clusters)')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inércia (WCSS)')
plt.xticks(possible_k_values)
plt.grid(axis='y')
plt.show()

In [ ]:
silhouette_scores_kmeans = []
# Silhouette score não é definido para K=1
possible_k_values_sil = range(2, 11)
for k_val in possible_k_values_sil:
    kmeans_temp = KMeans(n_clusters=k_val, init='k-means++', n_init='auto', random_state=42)
    cluster_labels_temp = kmeans_temp.fit_predict(X_seeds_scaled)
    silhouette_avg = silhouette_score(X_seeds_scaled, cluster_labels_temp)
    silhouette_scores_kmeans.append(silhouette_avg)
    print(f"Para K = {k_val}, o score médio de silhueta é: {silhouette_avg:.4f}")

In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(possible_k_values_sil, silhouette_scores_kmeans, marker='o', linestyle='-')
plt.title('Score médio de silhueta para diferentes K')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Score médio de silhueta')
plt.xticks(possible_k_values_sil)
plt.grid(axis='y')
plt.show()

In [ ]:
K_final_seeds = 2
kmeans_seeds = KMeans(n_clusters=K_final_seeds, init='k-means++', n_init='auto', random_state=42)
labels_kmeans_seeds = kmeans_seeds.fit_predict(X_seeds_scaled)
centroids_seeds = kmeans_seeds.cluster_centers_

In [ ]:
pca_seeds_viz = PCA(n_components=2)
X_seeds_pca_viz = pca_seeds_viz.fit_transform(X_seeds_scaled)

In [ ]:
plt.figure(figsize=(10, 7))

sns.scatterplot(x=X_seeds_pca_viz[:, 0], y=X_seeds_pca_viz[:, 1], hue=labels_kmeans_seeds, palette='viridis', s=100, alpha=0.7, legend='full')
centroids_pca_seeds = pca_seeds_viz.transform(centroids_seeds)
plt.scatter(centroids_pca_seeds[:, 0], centroids_pca_seeds[:, 1], marker='X', s=200, color='red', label='Centróides')
plt.title(f'Clusters K-Means (K={K_final_seeds})')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend()
plt.grid(False)
plt.show()

## Hierárquico

In [ ]:
from sklearn.datasets import load_wine
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.cluster import AgglomerativeClustering

#Para correlação cofenética
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import cophenet

from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

In [ ]:
wine = load_wine()
X_wine = wine.data
y_wine_true = wine.target #Guardar para avaliação de performance do modelo, não será usado no clustering

In [ ]:
#Padronização da escala
scaler_wine = StandardScaler()
X_wine_scaled = scaler_wine.fit_transform(X_wine)

In [ ]:
# Método de Ward é o padrão, os outros são: 'single', 'complete', 'average'
#metodo = 'single'
#metodo = 'complete'
#metodo = 'average'
metodo = 'ward'

linked_ward_wine = linkage(X_wine_scaled, method=metodo)

In [ ]:
plt.figure(figsize=(10, 7))

dendrogram(linked_ward_wine,
           orientation='top',
           distance_sort='descending',
           show_leaf_counts=True,
           truncate_mode='lastp', # Mostrar apenas os últimos p clusters fundidos
           p=12) # Número de clusters a mostrar nas folhas do dendrograma
plt.title('Cluster hierárquico (Dendrograma) - ' + metodo)
plt.xlabel('Índice da amostra (tamanho do cluster)')
plt.ylabel('Distância Euclidiana')
plt.grid(axis='y')
plt.show()

In [ ]:
num_clusters_wine = 3
labels_hier_scipy = fcluster(linked_ward_wine, t=num_clusters_wine, criterion='maxclust')
print(labels_hier_scipy)

In [ ]:
agg_cluster_wine = AgglomerativeClustering(n_clusters=num_clusters_wine, linkage='ward')
labels_hier_sklearn = agg_cluster_wine.fit_predict(X_wine_scaled)
print(labels_hier_sklearn)

In [ ]:
pca_wine_viz = PCA(n_components=2)
X_wine_pca_viz = pca_wine_viz.fit_transform(X_wine_scaled)

plt.figure(figsize=(10, 7))

sns.scatterplot(x=X_wine_pca_viz[:, 0], y=X_wine_pca_viz[:, 1], hue=labels_hier_sklearn, palette='viridis', s=100, alpha=0.7, legend='full')
plt.title(f'Clusters hierárquico (K={num_clusters_wine})')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Métricas de avaliação de performance
sil_score_hier = silhouette_score(X_wine_scaled, labels_hier_sklearn)
print(f"Hierárquico - Score de Silhueta: {sil_score_hier:.4f}")

In [ ]:
# Coeficiente de correlação cofenética
dist_orig_wine = pdist(X_wine_scaled)
dist_coph_wine = dendrogram(linked_ward_wine, no_plot=True) # Para obter as distâncias cofenéticas

In [ ]:
# A função cophenet é mais direta para isso
coph_corr_wine, _ = cophenet(linked_ward_wine, dist_orig_wine)
print(f"Hierárquico - Coeficiente de Correlação Cofenética: {coph_corr_wine:.4f}")

## Grafos

In [ ]:
#!pip install networkx

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.neighbors import kneighbors_graph
import matplotlib.patches as mpatches

from sklearn.cluster import SpectralClustering
from sklearn.datasets import make_circles, make_moons

In [ ]:
X_moons, y_moons_true = make_moons(n_samples=300, noise=0.05, random_state=42)
X_moons_scaled = StandardScaler().fit_transform(X_moons)
K_moons = 2 # Sabemos que são 2 luas

In [ ]:
n_neighbors = 10 # Exemplo: considerar os 10 vizinhos mais próximos
connectivity_matrix = kneighbors_graph(X_moons_scaled, n_neighbors=n_neighbors, mode='connectivity', include_self=False)


In [ ]:
adjacency_matrix = connectivity_matrix.maximum(connectivity_matrix.T) # Garante simetria (se i é vizinho de j, j também é vizinho de i no grafo visualizado)
G = nx.from_scipy_sparse_array(adjacency_matrix)


In [ ]:
spectral_moons = SpectralClustering(n_clusters=K_moons,
                                    affinity='rbf', # ou 'nearest_neighbors'
                                    gamma=1.0,
                                    random_state=42,
                                    assign_labels='kmeans') # ou 'discretize'

labels_spectral_moons = spectral_moons.fit_predict(X_moons_scaled)

In [ ]:
pos = {i: X_moons_scaled[i, :] for i in range(X_moons_scaled.shape[0])}
colors = [plt.cm.viridis(label / (K_moons - 1) if K_moons > 1 else 0) for label in labels_spectral_moons]

plt.figure(figsize=(10, 7))
nx.draw(G, pos, node_color=colors, node_size=100, with_labels=False,
        width=0.5, edge_color='grey', alpha=0.8) # edge_color e width para as arestas

# Adicionar o título e rótulos (que draw_networkx não faz por padrão)
plt.title(f'Spectral Clustering (K={K_moons}) em Dados Make_Moons com Grafo KNN (k={n_neighbors})')
plt.xlabel('Feature 1 (Escalada)')
plt.ylabel('Feature 2 (Escalada)')

# Criar uma legenda manual para as cores dos clusters
legend_patches = []
for i in range(K_moons):
    color = plt.cm.viridis(i / (K_moons - 1) if K_moons > 1 else 0)
    patch = mpatches.Patch(color=color, label=f'Cluster {i+1}')
    legend_patches.append(patch)
plt.legend(handles=legend_patches, loc='best')

plt.grid(axis='y')
plt.show()

Mudando a Matriz de Similaridade

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_moons_scaled[:, 0], y=X_moons_scaled[:, 1], hue=labels_spectral_moons, palette='viridis', s=100, alpha=0.8)
plt.title(f'Spectral Clustering (K={K_moons}) em Dados Make_Moons')
plt.xlabel('Feature 1 (Escalada)')
plt.ylabel('Feature 2 (Escalada)')
plt.legend()
plt.grid(axis='y')
plt.show()

In [ ]:
spectral_moons_knn = SpectralClustering(n_clusters=K_moons,
                                    affinity='nearest_neighbors',
                                    n_neighbors=10,
                                    random_state=42,
                                    assign_labels='kmeans') # ou 'discretize'

labels_spectral_moons_knn = spectral_moons_knn.fit_predict(X_moons_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_moons_scaled[:, 0], y=X_moons_scaled[:, 1], hue=labels_spectral_moons_knn, palette='viridis', s=100, alpha=0.8)
plt.title(f'Spectral Clustering (K={K_moons}) em Dados Make_Moons')
plt.xlabel('Feature 1 (Escalada)')
plt.ylabel('Feature 2 (Escalada)')
plt.legend()
plt.grid(axis='y')
plt.show()

In [ ]:
kmeans_moons_comp = KMeans(n_clusters=K_moons, init='k-means++', n_init='auto', random_state=42)
labels_kmeans_moons_comp = kmeans_moons_comp.fit_predict(X_moons_scaled)

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_moons_scaled[:, 0], y=X_moons_scaled[:, 1], hue=labels_kmeans_moons_comp, palette='viridis', s=100, alpha=0.8)
plt.title(f'K-Means (K={K_moons}) em Dados Make_Moons (Usando K-Means)')
plt.xlabel('Feature 1 (Escalada)')
plt.ylabel('Feature 2 (Escalada)')
plt.legend()
plt.grid(axis='y')
plt.show()

In [ ]:
sil_score_spectral = silhouette_score(X_moons_scaled, labels_spectral_moons)
print(f"Spectral (Moons) - Score de Silhueta: {sil_score_spectral:.4f}")

## DBScan

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors # Para o k-distance plot
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
iris_dbscan = load_iris()
X_iris_dbscan = iris_dbscan.data
y_iris_dbscan_true = iris_dbscan.target

In [ ]:
#Padronização da escala
scaler_iris_dbscan = StandardScaler()
X_iris_dbscan_scaled = scaler_iris_dbscan.fit_transform(X_iris_dbscan)

In [ ]:
min_samples_dbscan = 8
nbrs = NearestNeighbors(n_neighbors=min_samples_dbscan).fit(X_iris_dbscan_scaled)
distances_dbscan, indices_dbscan = nbrs.kneighbors(X_iris_dbscan_scaled)
k_distances_dbscan = np.sort(distances_dbscan[:, min_samples_dbscan-1])

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(k_distances_dbscan)
plt.title(f'K-Distance do DBScan (k={min_samples_dbscan})')
plt.xlabel("Pontos (ordenados por distância)")
plt.ylabel(f'{min_samples_dbscan}-ésima Distância do Vizinho (eps)')
plt.grid(axis='y')
plt.show()

In [ ]:
eps_iris_dbscan = 0.7
print(f"Valor de eps escolhido: {eps_iris_dbscan}")

In [ ]:
dbscan_iris_model = DBSCAN(eps=eps_iris_dbscan, min_samples=min_samples_dbscan)
labels_dbscan_iris = dbscan_iris_model.fit_predict(X_iris_dbscan_scaled)

In [ ]:
n_clusters_dbscan_ = len(set(labels_dbscan_iris)) - (1 if -1 in labels_dbscan_iris else 0)
n_noise_dbscan_ = list(labels_dbscan_iris).count(-1)
print(f'DBScan (Iris) - Número estimado de clusters: {n_clusters_dbscan_}')
print(f'DBScan (Iris) - Número estimado de pontos de ruído: {n_noise_dbscan_}')

In [ ]:
pca_iris_dbscan_viz = PCA(n_components=2)
X_iris_dbscan_pca_viz = pca_iris_dbscan_viz.fit_transform(X_iris_dbscan_scaled)

plt.figure(figsize=(10, 7))
plt.scatter(X_iris_dbscan_pca_viz[:, 0], X_iris_dbscan_pca_viz[:, 1], color='black', label='Iris')
plt.title(f'Pontos iniciais da Iris')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
pca_iris_dbscan_viz = PCA(n_components=2)
X_iris_dbscan_pca_viz = pca_iris_dbscan_viz.fit_transform(X_iris_dbscan_scaled)

plt.figure(figsize=(10, 7))
unique_labels_dbscan = set(labels_dbscan_iris)
colors_dbscan = plt.cm.get_cmap('Spectral', len(unique_labels_dbscan))
for k, col in zip(unique_labels_dbscan, colors_dbscan(np.linspace(0, 1, len(unique_labels_dbscan)))):
    if k == -1:
        col = [0, 0, 0, 1] # Preto para ruído

    class_member_mask = (labels_dbscan_iris == k)
    xy = X_iris_dbscan_pca_viz[class_member_mask]

    # Identificar core samples (se disponível e para diferenciar na plotagem)
    is_core_sample = np.zeros_like(labels_dbscan_iris, dtype=bool)
    if hasattr(dbscan_iris_model, 'core_sample_indices_') and len(dbscan_iris_model.core_sample_indices_) > 0:
        is_core_sample[dbscan_iris_model.core_sample_indices_] = True

    xy_core = X_iris_dbscan_pca_viz[class_member_mask & is_core_sample]
    xy_border = X_iris_dbscan_pca_viz[class_member_mask & ~is_core_sample]

    if k == -1:
        plt.plot(xy[:, 0], xy[:, 1], 'x', markerfacecolor=tuple(col), markeredgecolor='k', markersize=6, label='Ruído')
    else:
        plt.plot(xy_core[:, 0], xy_core[:, 1], 'o', markerfacecolor=tuple(col), markeredgecolor='k', markersize=10, label=f'Cluster {k} (Core)')
        plt.plot(xy_border[:, 0], xy_border[:, 1], 'o', markerfacecolor=tuple(col), markeredgecolor='k', markersize=6, label=f'Cluster {k} (Borda)')

plt.title(f'DBScan (eps={eps_iris_dbscan}, min_samples={min_samples_dbscan})')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
# Gerenciando legendas para evitar duplicatas
handles, legend_labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(legend_labels, handles))
plt.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.show()

In [ ]:
labels_no_noise_dbscan = labels_dbscan_iris[labels_dbscan_iris != -1]
data_no_noise_dbscan = X_iris_dbscan_scaled[labels_dbscan_iris != -1]

In [ ]:
sil_score_dbscan = silhouette_score(data_no_noise_dbscan, labels_no_noise_dbscan)
print(f"DBScan (Iris) - Score de Silhueta (sem contabilizar ruído): {sil_score_dbscan:.4f}")

# Redução de Dimensionalidade

## PCA - Análise de Componentes Principais

In [ ]:
from sklearn.datasets import load_wine # Usando Wine para PCA, pois tem mais features que Iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
wine_pca_data = load_wine()
X_wine_pca = wine_pca_data.data
y_wine_pca = wine_pca_data.target
feature_names_wine_pca = wine_pca_data.feature_names

In [ ]:
#Padronização da escala
scaler_wine_pca = StandardScaler()
X_wine_pca_scaled = scaler_wine_pca.fit_transform(X_wine_pca)

In [ ]:
# Especificar número de componentes OU Especificar variância desejada
pca_model = PCA(n_components=2) # Reduzir para 2 componentes para visualização
#pca_model = PCA(n_components=0.95) # Reter 95% da variância

In [ ]:
pca_model.fit(X_wine_pca_scaled)
cumulative_explained_variance = np.cumsum(pca_model.explained_variance_ratio_)

In [ ]:
# Scree Plot (Variância explicada por componente)
plt.figure(figsize=(10, 7))
plt.bar(range(1, len(pca_model.explained_variance_ratio_) + 1), pca_model.explained_variance_ratio_, alpha=0.7, align='center', label='Variância individual explicada')
plt.step(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, where='mid', label='Variância acumulada explicada')
plt.ylabel('Taxa de Variância Explicada')
plt.xlabel('Componentes Principais')
plt.title('PCA - Scree Plot')
plt.legend(loc='best')
plt.xticks(range(1, len(pca_model.explained_variance_ratio_) + 1))
plt.grid(axis='y')
plt.show()

In [ ]:
X_wine_transformed_pca = pca_model.fit_transform(X_wine_pca_scaled)
print(f"Dimensões originais: {X_wine_pca_scaled.shape}")
print(f"Dimensões reduzidas: {X_wine_transformed_pca.shape}")

In [ ]:
print(f"Variância explicada por cada componente: {pca_model.explained_variance_ratio_}")
cumulative_explained_variance = np.cumsum(pca_model.explained_variance_ratio_)
print(f"Variância explicada acumulada: {cumulative_explained_variance}")

In [ ]:
# Visualização dos dados transformados (se n_components=2)
df_pca_wine_viz = pd.DataFrame(data=X_wine_transformed_pca, columns=['PC1', 'PC2'])
df_pca_wine_viz['target'] = y_wine_pca

plt.figure(figsize=(10, 7))
sns.scatterplot(x='PC1', y='PC2', hue='target', data=df_pca_wine_viz, palette='viridis', s=100, alpha=0.8)
plt.title('PCA (2 Componentes)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Classe do Vinho')
plt.grid(axis='y')
plt.show()

In [ ]:
# Especificar número de componentes OU Especificar variância desejada
#pca_model = PCA(n_components=2) # Reduzir para 2 componentes para visualização
pca_model = PCA(n_components=0.95) # Reter 95% da variância

In [ ]:
pca_model.fit(X_wine_pca_scaled)
cumulative_explained_variance = np.cumsum(pca_model.explained_variance_ratio_)

In [ ]:
# Scree Plot (Variância explicada por componente)
plt.figure(figsize=(10, 7))
plt.bar(range(1, len(pca_model.explained_variance_ratio_) + 1), pca_model.explained_variance_ratio_, alpha=0.7, align='center', label='Variância individual explicada')
plt.step(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, where='mid', label='Variância acumulada explicada')
plt.ylabel('Taxa de Variância Explicada')
plt.xlabel('Componentes Principais')
plt.title('PCA - Scree Plot')
plt.legend(loc='best')
plt.xticks(range(1, len(pca_model.explained_variance_ratio_) + 1))
plt.grid(axis='y')
plt.show()

# Sistemas de Recomendação

## Market Basket Analysis com Apriori

In [ ]:
#!pip install mlxtend # (se não estiver instalado)

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

import pandas as pd

In [ ]:
# Dados de exemplo (transações)
transactions_market = [
    ['Leite', 'Pão', 'Açúcar', 'Café', 'Manteiga'],
    ['Mamão', 'Banana', 'Maça'],
    ['Leite', 'Pão'],
    ['Leite', 'Pão', 'Manteiga', 'Banana']
]


In [ ]:
# Transformar dados para o formato one-hot exigido pelo Apriori
te = TransactionEncoder()
te_ary = te.fit(transactions_market).transform(transactions_market)
df_market_trans = pd.DataFrame(te_ary, columns=te.columns_)
print("DataFrame de transações (One-hot encoded):")
print(df_market_trans.head())

In [ ]:
frequent_itemsets = apriori(df_market_trans, min_support=0.25, use_colnames=True) # Suporte de 25%
print("\nItemsets Frequentes (Suporte >= 0.25):")
print(frequent_itemsets.sort_values(by='support', ascending=False))

In [ ]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
print("\nRegras de Associação (Confiança >= 0.6):")
# Selecionar e ordenar colunas para melhor visualização
rules_sorted = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction']].sort_values(by='lift', ascending=False)
print(rules_sorted)



In [ ]:
rules.head(13)

Usando a base da UCI

In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
# Carregar o dataset
df = pd.read_excel('http://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx')

In [ ]:
# Remover espaços extras das colunas relevantes
df['Description'] = df['Description'].astype(str).str.strip()
# Aplicar o strip também ao InvoiceNo, pois ele é convertido para string
df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.strip()

# Remover transações canceladas (InvoiceNo começando com 'C')
df = df[~df['InvoiceNo'].str.contains('C')]

In [ ]:
df['Country'].unique()

In [ ]:
# Germany
pais = str(input("País: "))

basket = (df[df['Country'] == pais]
           .groupby(['InvoiceNo', 'Description'])['Quantity']
           .sum().unstack().reset_index().fillna(0)
           .set_index('InvoiceNo'))

In [ ]:
# Converter quantidades para formato binário (0 ou 1)
def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1

basket_sets = basket.applymap(encode_units)

In [ ]:
if 'POSTAGE' in basket_sets.columns:
    basket_sets.drop('POSTAGE', inplace=True, axis=1)

# Verificar o formato final do dataset para MBA
print("Formato do dataset para MBA (Alemanha):")
print(basket_sets.head())
print(f"Dimensões: {basket_sets.shape}")

In [ ]:
# Gerar conjuntos de itens frequentes com suporte mínimo de 7%
frequent_itemsets = apriori(basket_sets, min_support=0.07, use_colnames=True)

print("\nConjuntos de Itens Frequentes:")
print(frequent_itemsets.head())

In [ ]:
# Gerar regras de associação com lift mínimo de 1
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

print(rules)

In [ ]:
def recomendar_produtos(produto_base, regras_df, top_n=3):
    recomendacoes_potenciais = []

    if regras_df.empty:
        return

    # Iterar sobre as regras para encontrar aquelas onde o produto_base está no antecedente
    for _, rule in regras_df.iterrows():
        if produto_base in rule['antecedents']:
            for consequente_item in rule['consequents']:
                # Evitar recomendar o próprio produto_base
                if consequente_item!= produto_base:
                    recomendacoes_potenciais.append({
                        'item_recomendado': consequente_item,
                        'confidence': rule['confidence'],
                        'lift': rule['lift']
                    })

    if not recomendacoes_potenciais:
        return

    # Converter para DataFrame para facilitar a ordenação e remoção de duplicatas
    df_recomendacoes = pd.DataFrame(recomendacoes_potenciais)

    if df_recomendacoes.empty:
        return

    # Ordenar por lift (maior primeiro) e depois por confiança (maior primeiro)
    df_recomendacoes = df_recomendacoes.sort_values(by=['lift', 'confidence'], ascending=[False, False])

    # Obter itens recomendados únicos, mantendo a ordem da primeira aparição (melhor devido à ordenação)
    itens_unicos_recomendados = df_recomendacoes['item_recomendado'].unique()

    return list(itens_unicos_recomendados[:top_n])

In [ ]:
# Lift 1 / Confiança 0.01
lift = int(input("Digite o valor do lift: "))
confianca = float(input("Digite o valor da confiança: "))

filtered_rules = rules[
            (rules['lift'] >= lift) &
            (rules['confidence'] >= confianca)
        ]

filtered_rules_display = filtered_rules.copy()
filtered_rules_display['antecedents_sorted_tuple'] = filtered_rules_display['antecedents'].apply(lambda x: tuple(sorted(list(x))))

filtered_rules_display['antecedents_str'] = filtered_rules_display['antecedents'].astype(str)

result_list = filtered_rules_display.sort_values(by='antecedents_str')[[
    'antecedents',
    'consequents',
    'lift',
    'confidence' # Mantendo lift e confidence para referência do filtro
]]

print(f"\nRegras Filtradas com Lift >= {lift} e Confiança >= {confianca}, ordenadas por Antecedents:")
if not result_list.empty:
    for index, row in result_list.iterrows():
        antecedent_str = ", ".join(list(row['antecedents']))
        consequent_str = ", ".join(list(row['consequents']))
        print(f"SE {{{antecedent_str}}} ENTÃO {{{consequent_str}}} (Lift: {row['lift']:.2f}, Confiança: {row['confidence']:.2f})")
else:
    print("Nenhuma regra encontrada com os critérios de filtro especificados.")


In [ ]:
# Produto: "ROUND SNACK BOXES SET OF4 WOODLAND"
produto_especifico_teste = str(input("Nome do produto: "))

if not rules.empty and produto_especifico_teste in basket_sets.columns:
  recomendacoes_especificas = recomendar_produtos(produto_especifico_teste, rules, top_n=3)
  if recomendacoes_especificas:
    print(f"\nTop 3 produtos recomendados para quem compra '{produto_especifico_teste}':")
    for i, item in enumerate(recomendacoes_especificas):
      print(f"{i+1}. {item}")
  else:
    print(f"Não foi possível gerar recomendações para '{produto_especifico_teste}' com as regras atuais.")
elif rules.empty:
  print(f"\nNão há regras de associação para testar com '{produto_especifico_teste}'.")
elif produto_especifico_teste not in basket_sets.columns:
  print(f"\nProduto '{produto_especifico_teste}' não encontrado nas colunas do dataset para teste.")

# Detecção de Anomalia

## Isolation Forest


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.datasets import load_wine
from sklearn.ensemble import IsolationForest
from sklearn.tree import export_graphviz # Para visualização da árvore
import graphviz # Para visualização da árvore

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')


In [ ]:
# Carregar o dataset Wine
wine_anomaly_data = load_wine()
X_wine_anomaly = wine_anomaly_data.data


In [ ]:
#Padronização da escala
scaler_wine_anomaly = StandardScaler()
X_wine_anomaly_scaled = scaler_wine_anomaly.fit_transform(X_wine_anomaly)

In [ ]:
# contamination: proporção esperada de outliers no conjunto de dados
iso_forest = IsolationForest(n_estimators=100, contamination='auto', random_state=42) # 'auto' permite que o modelo decida
#iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42) # definir um valor (ex: 0.05 para 5% de anomalias)

In [ ]:
iso_forest.fit(X_wine_anomaly_scaled)
predictions_anomaly = iso_forest.predict(X_wine_anomaly_scaled)

In [ ]:
df_wine_anomaly_results = pd.DataFrame(X_wine_anomaly_scaled, columns=wine_anomaly_data.feature_names)
df_wine_anomaly_results['anomaly_score'] = iso_forest.decision_function(X_wine_anomaly_scaled) # Scores mais baixos são mais anômalos
df_wine_anomaly_results['is_anomaly_predicted'] = predictions_anomaly
df_wine_anomaly_results['is_anomaly_predicted'] = df_wine_anomaly_results['is_anomaly_predicted'].map({1: 'Inlier', -1: 'Outlier'})

In [ ]:
print("Resultado da Detecção de Anomalias (Isolation Forest):")
print(f"Número de outliers detectados: {sum(predictions_anomaly == -1)}")
print(f"Número de inliers detectados: {sum(predictions_anomaly == 1)}")
print("------------")
print("Primeiras 10 amostras com seus scores e predições de anomalia:")
print(df_wine_anomaly_results[['anomaly_score', 'is_anomaly_predicted']].head(10))
print("------------")
print("Primeiras 10 amostras  consideradas outliers:")
print(df_wine_anomaly_results[df_wine_anomaly_results['is_anomaly_predicted'] == 'Outlier'])

In [ ]:
pca_anomaly_viz = PCA(n_components=2)
X_wine_anomaly_pca_viz = pca_anomaly_viz.fit_transform(X_wine_anomaly_scaled)

plt.figure(figsize=(10, 7))
# Plotar inliers
plt.scatter(X_wine_anomaly_pca_viz[predictions_anomaly == 1, 0],
            X_wine_anomaly_pca_viz[predictions_anomaly == 1, 1],
            c='blue', label='Inlier', alpha=0.6, s=50)
# Plotar outliers
plt.scatter(X_wine_anomaly_pca_viz[predictions_anomaly == -1, 0],
            X_wine_anomaly_pca_viz[predictions_anomaly == -1, 1],
            c='red', marker='x', label='Outlier (Anomalia)', s=100)

plt.title('Detecção de Anomalias com Isolation Forest (Dataset Wine - PCA)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend()
plt.grid(axis='y')
plt.show()